# 06 - Explainability & Interpretability

This notebook provides model interpretation for publication:
1. Load final model and selected features
2. Compute feature importance
3. Generate biomarker ranking
4. Run SHAP analysis (Summary, Waterfall, Dependence plots)
5. Save all figures and tables

> **Note:** All logic is in `src/explainability.py`. This notebook only orchestrates.

In [29]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import pandas as pd
import matplotlib.pyplot as plt

import config
from src.io import load_model, save_figure, save_table, logger
from src.models import xgb_safe_frame, xgb_feature_name_map
from src.explainability import (
    compute_feature_importance,
    get_biomarker_ranking,
    run_explainability,
)
from src.visualization import setup_style, plot_feature_importance

setup_style()

In [30]:
import sys
from pathlib import Path
import pandas as pd
import joblib

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
import config

# 1. Load Model
model = joblib.load(config.MODELS_DIR / "best_model_xgboost.joblib")

# 2. Load the FINAL feature list (که شامل ویژگی‌های مهندسی شده است)
# توجه: نام فایل را به selected_features_final.csv تغییر دادیم
try:
    features_df = pd.read_csv(config.TABLES_DIR / "selected_features_final.csv")
    selected_features = features_df["feature"].tolist()
    print(f"Loaded {len(selected_features)} features (including engineered ones).")
except FileNotFoundError:
    print("⚠️ Warning: 'selected_features_final.csv' not found. Falling back to old list.")
    features_df = pd.read_csv(config.TABLES_DIR / "selected_features.csv")
    selected_features = features_df["feature"].tolist()

# 3. Load Test Data (که قبلاً در نوت‌بوک ۰۵ ذخیره کردیم و شامل ستون‌های جدید است)
X_test = pd.read_csv(config.PROCESSED_DIR / "X_test_selected.csv")
y_test = pd.read_csv(config.PROCESSED_DIR / "y_test.csv").iloc[:, 0]

# Sanity Check
if len(selected_features) != X_test.shape[1]:
    print(f"⚠️ Mismatch: Features list ({len(selected_features)}) vs Data columns ({X_test.shape[1]})")
else:
    print(f"✅ Data aligned: {X_test.shape[1]} features.")

# نمایش چند ویژگی اول برای اطمینان
print("First 5 features:", selected_features[:5])

Loaded 37 features (including engineered ones).
✅ Data aligned: 37 features.
First 5 features: ['SHC2', 'OAZ2', 'PRAG1', 'SHMT1', 'TRIM7']


## Step 1: Load Final Model and Selected Features

In [31]:
# # Load the best model (from Model Training notebook)
# model = load_model("best_model_xgboost.joblib")

# # Load selected features
# selected_df = pd.read_csv(config.TABLES_DIR / "selected_features.csv")
# selected_features = selected_df["feature"].tolist()

# # Load test data
# X_test_selected = pd.read_csv(config.PROCESSED_DIR / "X_test_selected.csv")
# y_test = pd.read_csv(config.PROCESSED_DIR / "y_test.csv").iloc[:, 0]

# print(f"Model loaded: {type(model).__name__}")
# print(f"Selected features: {len(selected_features)}")
# print(f"Test samples: {len(X_test_selected)}")

## Step 2: Feature Importance

In [32]:
# Map XGBoost-safe names back to original feature names
name_map = xgb_feature_name_map(selected_features)

importance_df = compute_feature_importance(
    model,
    selected_features,
    top_k=20,
)

print("Top 20 Features by Importance:")
print(importance_df.to_string(index=False))

2026-08-14 21:01:14 | INFO     | prostate_bcr | Extracted feature importance: 37 features


Top 20 Features by Importance:
                                             feature  importance
Primary Lymph Node Presentation Assessment Ind-3_YES    0.074959
                                  Margin_x_LymphNode    0.066126
                                       Gleason_Total    0.064840
                                              DYNLT1    0.058162
                                               PRAG1    0.058010
                                              PTGER1    0.036174
                                               AS3MT    0.031097
                                  AR_Signaling_Score    0.030994
                                               FGF20    0.027936
                                               AIFM3    0.027176
                                               CNTRL    0.026806
                                               SEPT1    0.026609
                                                SHC2    0.025770
                                                PIM2    0.0

In [33]:
fig = plot_feature_importance(
    importance_df,
    top_k=20,
    title="Top 20 Features — XGBoost Importance",
    filename="feature_importance_top20.png",
)
plt.show()

2026-08-14 21:01:15 | INFO     | prostate_bcr | Saved figure → D:\Prostate_BCR\core\outputs\figures\feature_importance_top20.png
2026-08-14 21:01:15 | INFO     | prostate_bcr | Feature importance plot generated (20 features)


## Step 3: Biomarker Ranking

Separate clinical and gene features for biomarker analysis.

In [34]:
from src.preprocessing import identify_column_groups

clinical_cols, gene_cols = identify_column_groups(X_test_selected)

# Rank gene biomarkers
gene_importance = importance_df[importance_df["feature"].isin(gene_cols)].copy()
gene_ranking = get_biomarker_ranking(gene_importance, feature_type="gene")

# Rank clinical biomarkers
clinical_importance = importance_df[importance_df["feature"].isin(clinical_cols)].copy()
clinical_ranking = get_biomarker_ranking(clinical_importance, feature_type="clinical")

print("Top Gene Biomarkers:")
print(gene_ranking.head(10).to_string(index=False))

print("\nTop Clinical Biomarkers:")
print(clinical_ranking.head(10).to_string(index=False))

2026-08-14 21:01:15 | INFO     | prostate_bcr | Column groups: 1 clinical, 29 gene
2026-08-14 21:01:15 | INFO     | prostate_bcr | Biomarker ranking: 15 gene features
2026-08-14 21:01:15 | INFO     | prostate_bcr | Biomarker ranking: 1 clinical features


Top Gene Biomarkers:
 rank feature  importance feature_type
    1  DYNLT1    0.058162         gene
    2   PRAG1    0.058010         gene
    3  PTGER1    0.036174         gene
    4   AS3MT    0.031097         gene
    5   FGF20    0.027936         gene
    6   AIFM3    0.027176         gene
    7   CNTRL    0.026806         gene
    8   SEPT1    0.026609         gene
    9    SHC2    0.025770         gene
   10    PIM2    0.025727         gene

Top Clinical Biomarkers:
 rank                                              feature  importance feature_type
    1 Primary Lymph Node Presentation Assessment Ind-3_YES    0.074959     clinical


In [35]:
save_table(gene_ranking, "biomarker_ranking_genes.csv", index=False)
save_table(clinical_ranking, "biomarker_ranking_clinical.csv", index=False)
save_table(importance_df, "feature_importance_full.csv", index=False)

print("Biomarker rankings saved to outputs/tables/")

2026-08-14 21:01:15 | INFO     | prostate_bcr | Saved 15 rows → D:\Prostate_BCR\core\outputs\tables\biomarker_ranking_genes.csv
2026-08-14 21:01:15 | INFO     | prostate_bcr | Saved 1 rows → D:\Prostate_BCR\core\outputs\tables\biomarker_ranking_clinical.csv
2026-08-14 21:01:15 | INFO     | prostate_bcr | Saved 20 rows → D:\Prostate_BCR\core\outputs\tables\feature_importance_full.csv


Biomarker rankings saved to outputs/tables/


## Step 4: SHAP Analysis

> **Note:** SHAP analysis requires the `shap` package.
> If not installed, run: `pip install shap`

In [36]:
# Run full explainability pipeline with SHAP
results = run_explainability(
    model=model,
    X_test=X_test_selected,
    feature_names=selected_features,
    top_k=20,
    run_shap=True,
    sample_index=0,
)

if "shap_error" in results:
    print(f"SHAP analysis skipped: {results['shap_error']}")
else:
    print("SHAP analysis completed successfully.")

2026-08-14 21:01:15 | INFO     | prostate_bcr | Extracted feature importance: 37 features
2026-08-14 21:01:15 | INFO     | prostate_bcr | Biomarker ranking: 20 gene features
2026-08-14 21:01:15 | WARNING  | prostate_bcr | SHAP analysis failed: [21:01:15] C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api_utils.h:129: Check failed: std::accumulate(shape.cbegin(), shape.cend(), static_cast<bst_ulong>(1), std::multiplies<>{}) == chunksize * rows (3999 vs. 4902) : 
2026-08-14 21:01:15 | INFO     | prostate_bcr | Explainability pipeline complete


SHAP analysis skipped: [21:01:15] C:\actions-runner\_work\xgboost\xgboost\src\c_api\c_api_utils.h:129: Check failed: std::accumulate(shape.cbegin(), shape.cend(), static_cast<bst_ulong>(1), std::multiplies<>{}) == chunksize * rows (3999 vs. 4902) : 


In [37]:
if "summary_plot" in results:
    fig = results["summary_plot"]
    save_figure(fig, "shap_summary_plot.png")
    plt.show()
else:
    print("SHAP summary plot not available.")

SHAP summary plot not available.


In [38]:
if "waterfall_plot" in results:
    fig = results["waterfall_plot"]
    save_figure(fig, "shap_waterfall_sample0.png")
    plt.show()
else:
    print("SHAP waterfall plot not available.")

SHAP waterfall plot not available.


## Step 5: SHAP Dependence Plot (Top Feature)

In [39]:
if "shap_values" in results:
    from src.explainability import plot_shap_dependence

    top_feature = importance_df.iloc[0]["feature"]
    fig = plot_shap_dependence(
        results["shap_values"],
        X_test_selected,
        feature_name=top_feature,
        figsize=(8, 6),
    )
    save_figure(fig, f"shap_dependence_{top_feature[:20]}.png")
    plt.show()
    print(f"Dependence plot generated for: {top_feature}")
else:
    print("SHAP values not available.")

SHAP values not available.


## Summary

| Output | Location |
|--------|----------|
| Feature Importance Table | `outputs/tables/feature_importance_full.csv` |
| Gene Biomarker Ranking | `outputs/tables/biomarker_ranking_genes.csv` |
| Clinical Biomarker Ranking | `outputs/tables/biomarker_ranking_clinical.csv` |
| Feature Importance Plot | `outputs/figures/feature_importance_top20.png` |
| SHAP Summary Plot | `outputs/figures/shap_summary_plot.png` |
| SHAP Waterfall Plot | `outputs/figures/shap_waterfall_sample0.png` |
| SHAP Dependence Plot | `outputs/figures/shap_dependence_*.png` |



In [40]:
# Save the final feature list (including engineered ones)
pd.DataFrame({"feature": selected_features}).to_csv(
    config.TABLES_DIR / "selected_features_final.csv", index=False
)
print("Saved selected_features_final.csv")

Saved selected_features_final.csv
